In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EuroSAT
from torchvision import models
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from sklearn.metrics import classification_report
import numpy as np
import json, time, random, copy

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
assert str(device) == "cuda", "Switch to T4 GPU! Runtime → Change runtime type → T4 GPU"

Device: cuda


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

full_dataset = EuroSAT(root="./data", download=True, transform=train_transform)
CLASS_NAMES  = full_dataset.classes
print(f"Total images : {len(full_dataset)}")
print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")

total   = len(full_dataset)
n_train = int(0.70 * total)
n_val   = int(0.15 * total)
n_test  = total - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

val_dataset  = Subset(full_dataset, val_ds.indices)
test_dataset = Subset(full_dataset, test_ds.indices)

train_loader = DataLoader(train_ds,     batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

100%|██████████| 94.3M/94.3M [00:01<00:00, 67.9MB/s]


Total images : 27000
Classes (10): ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train: 18900 | Val: 4050 | Test: 4050


In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(512, len(CLASS_NAMES))
)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 91.1MB/s]


Trainable parameters: 5,130


In [ ]:
criterion = nn.CrossEntropyLoss()

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(imgs)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += len(imgs)
    return total_loss / total, correct / total

# Phase 1 — head only
print("Phase 1: Training head only (3 epochs)...")
opt1 = optim.Adam(model.fc.parameters(), lr=1e-3)
for ep in range(3):
    tr_loss, tr_acc   = run_epoch(model, train_loader, opt1)
    val_loss, val_acc = run_epoch(model, val_loader)
    print(f"  Ep {ep+1}/3 | loss {tr_loss:.3f} | train {tr_acc:.3f} | val {val_acc:.3f}")

# Phase 2 — full fine-tune
print("\nPhase 2: Fine-tuning full model (7 epochs)...")
for param in model.parameters():
    param.requires_grad = True

opt2  = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=7)

best_acc, best_weights = 0.0, None
for ep in range(7):
    tr_loss, tr_acc   = run_epoch(model, train_loader, opt2)
    val_loss, val_acc = run_epoch(model, val_loader)
    sched.step()
    flag = " ← best" if val_acc > best_acc else ""
    print(f"  Ep {ep+1}/7 | loss {tr_loss:.3f} | train {tr_acc:.3f} | val {val_acc:.3f}{flag}")
    if val_acc > best_acc:
        best_acc     = val_acc
        best_weights = copy.deepcopy(model.state_dict())

model.load_state_dict(best_weights)
print(f"\nBest val accuracy: {best_acc:.4f}")

Phase 1: Training head only (3 epochs)...
  Ep 1/3 | loss 1.050 | train 0.669 | val 0.805
  Ep 2/3 | loss 0.714 | train 0.773 | val 0.839
  Ep 3/3 | loss 0.682 | train 0.780 | val 0.827

Phase 2: Fine-tuning full model (7 epochs)...
  Ep 1/7 | loss 0.312 | train 0.899 | val 0.948 ← best
  Ep 2/7 | loss 0.161 | train 0.946 | val 0.958 ← best
  Ep 3/7 | loss 0.123 | train 0.959 | val 0.967 ← best
  Ep 4/7 | loss 0.088 | train 0.970 | val 0.968 ← best
  Ep 5/7 | loss 0.067 | train 0.977 | val 0.972 ← best
  Ep 6/7 | loss 0.059 | train 0.981 | val 0.969
  Ep 7/7 | loss 0.048 | train 0.984 | val 0.971

Best val accuracy: 0.9716


In [ ]:
model.eval()
all_preds, all_labels, latencies = [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        t0   = time.perf_counter()
        out  = model(imgs)
        t1   = time.perf_counter()
        latencies.append(((t1 - t0) / len(imgs)) * 1000)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))
avg_lat = float(np.mean(latencies))
print(f"Avg inference latency per image: {avg_lat:.2f} ms")

report = classification_report(all_labels, all_preds,
                                target_names=CLASS_NAMES, output_dict=True)
metrics_summary = {
    "accuracy"       : round(report["accuracy"], 4),
    "precision"      : round(report["macro avg"]["precision"], 4),
    "recall"         : round(report["macro avg"]["recall"], 4),
    "f1"             : round(report["macro avg"]["f1-score"], 4),
    "avg_latency_ms" : round(avg_lat, 2),
    "per_class"      : {
        cls: {
            "precision": round(report[cls]["precision"], 4),
            "recall"   : round(report[cls]["recall"], 4),
            "f1"       : round(report[cls]["f1-score"], 4),
        } for cls in CLASS_NAMES
    }
}
with open("metrics_summary.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)
print("\nSaved: metrics_summary.json")
print(json.dumps({k: v for k, v in metrics_summary.items() if k != "per_class"}, indent=2))

                      precision    recall  f1-score   support

          AnnualCrop       0.97      0.97      0.97       472
              Forest       0.98      1.00      0.99       442
HerbaceousVegetation       0.97      0.97      0.97       458
             Highway       0.96      0.96      0.96       391
          Industrial       0.97      0.99      0.98       378
             Pasture       0.96      0.95      0.95       299
       PermanentCrop       0.97      0.96      0.96       379
         Residential       1.00      0.98      0.99       450
               River       0.95      0.95      0.95       375
             SeaLake       0.99      0.99      0.99       406

            accuracy                           0.97      4050
           macro avg       0.97      0.97      0.97      4050
        weighted avg       0.97      0.97      0.97      4050

Avg inference latency per image: 0.16 ms

Saved: metrics_summary.json
{
  "accuracy": 0.9721,
  "precision": 0.9713,
  "recall": 

In [ ]:
DECISION_THRESHOLDS = {
    "discard": 30, "store": 60, "compress": 80,
    "cloud_veto": 0.85, "compression_ratio": 0.18
}
SCENE_PRIOR = {
    "AnnualCrop": 0.5, "Forest": 0.62, "HerbaceousVegetation": 0.4,
    "Highway": 0.7, "Industrial": 0.82, "Pasture": 0.42,
    "PermanentCrop": 0.5, "Residential": 0.78,
    "River": 0.74, "SeaLake": 0.55
}

model.eval()
frames     = []
sample_idx = random.sample(range(len(test_dataset)), 200)
rng        = np.random.default_rng(42)

for orbit_sec, idx in enumerate(sample_idx):
    img_tensor, true_label = test_dataset[idx]
    inp = img_tensor.unsqueeze(0).to(device)

    t0 = time.perf_counter()
    with torch.no_grad():
        logits = model(inp)
    t1 = time.perf_counter()

    probs      = torch.softmax(logits, dim=1)[0].cpu().numpy()
    pred_idx   = int(probs.argmax())
    pred_class = CLASS_NAMES[pred_idx]
    confidence = float(probs[pred_idx])

    cloud          = float(rng.beta(1.5, 5))
    novelty        = float(rng.uniform(0.2, 0.9))
    object_density = confidence

    prior         = SCENE_PRIOR[pred_class]
    base          = 100 * (0.45 * prior + 0.25 * object_density + 0.20 * novelty)
    cloud_penalty = 100 * 0.35 * cloud
    relevance     = int(max(0, min(100, base - cloud_penalty + (1 - cloud) * 8)))

    t = DECISION_THRESHOLDS
    if   cloud >= t["cloud_veto"]  : decision = "DISCARD"
    elif relevance <= t["discard"] : decision = "DISCARD"
    elif relevance <= t["store"]   : decision = "STORE"
    elif relevance <= t["compress"]: decision = "COMPRESS"
    else                           : decision = "TRANSMIT"

    raw_kb = 12288
    tx_kb  = (0 if decision in ("DISCARD", "STORE")
              else raw_kb * t["compression_ratio"] if decision == "COMPRESS"
              else raw_kb)

    frames.append({
        "id"                : f"F{orbit_sec:04d}",
        "ts"                : int(time.time() * 1000) + orbit_sec * 500,
        "orbitSec"          : orbit_sec + 1,
        "lat"               : round(float(rng.uniform(-60, 60)), 4),
        "lon"               : round(float(rng.uniform(-180, 180)), 4),
        "sceneClass"        : pred_class,
        "trueClass"         : CLASS_NAMES[true_label],
        "confidence"        : round(confidence, 4),
        "cloudCover"        : round(cloud, 3),
        "objectDensity"     : round(object_density, 3),
        "novelty"           : round(novelty, 3),
        "relevance"         : relevance,
        "decision"          : decision,
        "rawSizeKB"         : raw_kb,
        "transmittedSizeKB" : round(tx_kb, 1),
        "latencyMs"         : round((t1 - t0) * 1000, 2),
        "thumbHue"          : int(pred_idx * 36),
        "correct"           : pred_class == CLASS_NAMES[true_label],
    })

with open("real_frames.json", "w") as f:
    json.dump(frames, f, indent=2)

correct_count = sum(f["correct"] for f in frames)
decisions     = {d: sum(1 for f in frames if f["decision"] == d)
                 for d in ("TRANSMIT", "COMPRESS", "STORE", "DISCARD")}
print(f"Exported: real_frames.json ({len(frames)} frames)")
print(f"Correct predictions : {correct_count}/200")
print(f"Decision breakdown  : {decisions}")

Exported: real_frames.json (200 frames)
Correct predictions : 190/200
Decision breakdown  : {'TRANSMIT': 2, 'COMPRESS': 103, 'STORE': 92, 'DISCARD': 3}


In [ ]:
from google.colab import files
torch.save(model.state_dict(), "orbital_ai_resnet18.pth")
files.download("real_frames.json")
files.download("metrics_summary.json")
files.download("orbital_ai_resnet18.pth")
print("Done! Check your downloads folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Check your downloads folder.


In [ ]:
import os, json, base64
from PIL import Image
from google.colab import files

# Load existing real_frames.json
with open("real_frames.json") as f:
    frames = json.load(f)

# Create images folder
os.makedirs("eurosat_thumbs", exist_ok=True)

# Save each frame's image as JPG
model.eval()
saved = []

for frame in frames:
    idx = sample_idx[int(frame["orbitSec"]) - 1]
    img_tensor, _ = test_dataset[idx]

    # Denormalize — reverse the ImageNet normalization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img_denorm = img_tensor * std + mean
    img_denorm = img_denorm.clamp(0, 1)

    # Convert to PIL and save
    pil_img = transforms.ToPILImage()(img_denorm)
    pil_img = pil_img.resize((128, 128), Image.LANCZOS)

    fname = f"{frame['id']}.jpg"
    pil_img.save(f"eurosat_thumbs/{fname}", quality=85)

    # Add image path to frame
    frame["imagePath"] = f"/images/{fname}"
    saved.append(fname)

# Save updated real_frames.json
with open("real_frames.json", "w") as f:
    json.dump(frames, f, indent=2)

print(f"✅ Saved {len(saved)} images")
print("Sample frame:", json.dumps(frames[0], indent=2))

✅ Saved 200 images
Sample frame: {
  "id": "F0000",
  "ts": 1782401177177,
  "orbitSec": 1,
  "lat": 57.0747,
  "lon": 94.0103,
  "sceneClass": "Highway",
  "trueClass": "River",
  "confidence": 0.7185,
  "cloudCover": 0.191,
  "objectDensity": 0.718,
  "novelty": 0.266,
  "relevance": 54,
  "decision": "STORE",
  "rawSizeKB": 12288,
  "transmittedSizeKB": 0,
  "latencyMs": 31.91,
  "thumbHue": 108,
  "correct": false,
  "imagePath": "/images/F0000.jpg"
}


In [ ]:
import os
import time
import random
import json
import numpy as np
import torch
from torchvision import transforms
from PIL import Image
from collections import deque
SCENE_PRIOR = {
    "AnnualCrop": 0.5, "Forest": 0.62, "HerbaceousVegetation": 0.4,
    "Highway": 0.7, "Industrial": 0.82, "Pasture": 0.42,
    "PermanentCrop": 0.5, "Residential": 0.78,
    "River": 0.74, "SeaLake": 0.55
}
DECISION_THRESHOLDS = {
    "discard": 30, "store": 60, "compress": 80,
    "cloud_veto": 0.85, "compression_ratio": 0.18
}

os.makedirs("eurosat_thumbs", exist_ok=True)
model.eval()
frames = []
rng    = np.random.default_rng(42)
recent_classes = deque(maxlen=10)

N = 2000
sample_idx = random.sample(range(len(test_dataset)), N)
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

print(f"Exporting {N} frames + images...")
for orbit_sec, idx in enumerate(sample_idx):
    if orbit_sec % 200 == 0:
        print(f"  {orbit_sec}/{N}...")

    img_tensor, true_label = test_dataset[idx]
    inp = img_tensor.unsqueeze(0).to(device)

    t0 = time.perf_counter()
    with torch.no_grad():
        logits = model(inp)
    t1 = time.perf_counter()

    probs      = torch.softmax(logits, dim=1)[0].cpu().numpy()
    pred_idx   = int(probs.argmax())
    pred_class = CLASS_NAMES[pred_idx]
    confidence = float(probs[pred_idx])

    # Save thumbnail
    img_denorm = (img_tensor * std + mean).clamp(0, 1)
    pil_img    = transforms.ToPILImage()(img_denorm)
    pil_img    = pil_img.resize((128, 128), Image.LANCZOS)
    fname      = f"F{orbit_sec:04d}.jpg"
    pil_img.save(f"eurosat_thumbs/{fname}", quality=85)

    # Real novelty from class frequency
    novelty = float(1.0 - (recent_classes.count(pred_class) / max(len(recent_classes), 1)))
    recent_classes.append(pred_class)

    # Real cloud from classifier uncertainty
    cloud = float(np.clip(1.0 - confidence, 0.05, 0.90)) \
            if confidence < 0.85 \
            else float(rng.beta(1.5, 8))

    # Real object density from scene prior × confidence
    object_density = float(np.clip(
        SCENE_PRIOR[pred_class] * confidence * float(rng.uniform(0.85, 1.0)),
        0, 1
    ))

    # European coordinates (EuroSAT acquisition region)
    lat = round(float(rng.uniform(37, 65)), 4)
    lon = round(float(rng.uniform(-10, 40)), 4)

    prior         = SCENE_PRIOR[pred_class]
    base          = 100 * (0.45 * prior + 0.25 * object_density + 0.20 * novelty)
    cloud_penalty = 100 * 0.35 * cloud
    relevance     = int(max(0, min(100, base - cloud_penalty + (1 - cloud) * 8)))

    t = DECISION_THRESHOLDS
    if   cloud >= t["cloud_veto"]  : decision = "DISCARD"
    elif relevance <= t["discard"] : decision = "DISCARD"
    elif relevance <= t["store"]   : decision = "STORE"
    elif relevance <= t["compress"]: decision = "COMPRESS"
    else                           : decision = "TRANSMIT"

    raw_kb = 12288
    tx_kb  = (0 if decision in ("DISCARD", "STORE")
              else raw_kb * t["compression_ratio"] if decision == "COMPRESS"
              else raw_kb)

    frames.append({
        "id"                : f"F{orbit_sec:04d}",
        "ts"                : int(time.time() * 1000) + orbit_sec * 500,
        "orbitSec"          : orbit_sec + 1,
        "lat"               : lat,
        "lon"               : lon,
        "sceneClass"        : pred_class,
        "trueClass"         : CLASS_NAMES[true_label],
        "confidence"        : round(confidence, 4),
        "cloudCover"        : round(cloud, 3),
        "objectDensity"     : round(object_density, 3),
        "novelty"           : round(novelty, 3),
        "relevance"         : relevance,
        "decision"          : decision,
        "rawSizeKB"         : raw_kb,
        "transmittedSizeKB" : round(tx_kb, 1),
        "latencyMs"         : round((t1 - t0) * 1000, 2),
        "thumbHue"          : int(pred_idx * 36),
        "imagePath"         : f"/images/{fname}",
        "correct"           : pred_class == CLASS_NAMES[true_label],
    })

with open("real_frames.json", "w") as f:
    json.dump(frames, f, indent=2)

correct   = sum(f["correct"] for f in frames)
decisions = {d: sum(1 for f in frames if f["decision"] == d)
             for d in ("TRANSMIT","COMPRESS","STORE","DISCARD")}
print(f"\n✅ {N} frames exported")
print(f"Accuracy on export: {correct}/{N} ({100*correct/N:.1f}%)")
print(f"Decisions: {decisions}")

Exporting 2000 frames + images...
  0/2000...
  200/2000...
  400/2000...
  600/2000...
  800/2000...
  1000/2000...
  1200/2000...
  1400/2000...
  1600/2000...
  1800/2000...

✅ 2000 frames exported
Accuracy on export: 1939/2000 (97.0%)
Decisions: {'TRANSMIT': 17, 'COMPRESS': 872, 'STORE': 1103, 'DISCARD': 8}


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("eurosat_thumbs", "zip", "eurosat_thumbs")
size_mb = round(os.path.getsize("eurosat_thumbs.zip") / 1024 / 1024, 1)
print(f"Zip size: {size_mb} MB")

files.download("real_frames.json")
files.download("eurosat_thumbs.zip")
files.download("oracle_resnet18.pth")
print("✅ All downloaded!")

Zip size: 5.5 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FileNotFoundError: Cannot find file: oracle_resnet18.pth